# Phase 1 : Définition du problème et plan de collecte de données

## 1. Énoncé du problème

L'objectif de ce projet est de prédire la probabilité qu'un emprunteur ne rembourse pas son crédit — la variable cible étant le **statut de défaut (0 pour un remboursement complet, 1 pour un défaut de paiement)** —, afin d'éclairer et d'optimiser la décision d'octroi ou de refus de prêt par l'institution financière.

---

## 2. Types de données à collecter

Pour bâtir un modèle prédictif performant, nous devons segmenter et analyser plusieurs catégories de variables :

* **Données sociodémographiques et situation professionnelle :** Ces informations (âge, profession, type de contrat, ancienneté, situation matrimoniale) permettent d'évaluer la stabilité globale de la situation de l'emprunteur et la régularité de ses rentrées d'argent futures.
* **Données financières et de revenus :** Le niveau de revenus mensuels, le ratio d'endettement actuel et le reste à vivre sont essentiels pour mesurer directement la capacité financière de l'emprunteur à supporter une nouvelle mensualité.
* **Historique de crédit et scores de comportement :** Le score de crédit (ex. score FICO ou Bureau de crédit) et les antécédents de paiement (retards de paiement passés, incidents de remboursement) constituent les indicateurs les plus puissants du comportement de paiement futur d'un individu.
* **Caractéristiques du prêt demandé :** Le montant du prêt, sa durée, le taux d'intérêt et l'évaluation de la garantie (collatéral) influencent directement le niveau de risque ; un montant trop élevé par rapport aux revenus augmente mécaniquement la probabilité de défaut.

---

## 3. Sources de données et intégration

Pour alimenter notre modèle, nous devons croiser des sources internes et externes de manière réaliste :

* **Système d'information interne de l'institution financière (Core Banking) :** Cette source contient l'historique des transactions, les soldes des comptes et les caractéristiques des prêts passés de nos clients existants ; nous l'intégrerons via des requêtes SQL directes sur l'entrepôt de données (Data Warehouse) de la banque.
* **Bureaux de crédit (Credit Bureaus / Centrales des risques) :** Ces organismes centralisent l'historique d'endettement global d'un individu auprès de toutes les institutions de la place ; l'accès se fera par le biais d'intégrations d'API sécurisées interrogées en temps réel lors de la soumission de la demande.
* **Formulaires de demande de prêt en ligne (Données déclaratives) :** Il s'agit des données saisies directement par l'utilisateur lors de sa souscription (revenus déclarés, motif du prêt) ; elles seront extraites des bases de données applicatives (NoSQL ou SQL) liées à la plateforme web ou mobile.

---

## 4. Risques, contraintes et gouvernance

Le succès de ce projet repose sur la gestion stricte de plusieurs risques techniques et réglementaires. Sur le plan juridique, la collecte et le traitement de données financières hautement sensibles imposent une conformité rigoureuse avec les réglementations sur la protection des données (telles que le RGPD ou les lois nationales équivalentes), exigeant l'anonymisation des données personnelles (PII) et le cryptage des flux. Du point de vue de la qualité des données, nous serons confrontés à des défis majeurs tels que les valeurs manquantes, les erreurs de saisie manuelle et le déséquilibre des classes (les cas de défaut étant heureusement minoritaires par rapport aux bons payeurs), ce qui nécessitera des techniques d'échantillonnage adaptées. De plus, les biais d'échantillonnage — notamment le fait de ne posséder l'historique de remboursement que pour les clients ayant *déjà* obtenu un prêt par le passé — peuvent fausser les prédictions sur les nouveaux profils. Enfin, une gouvernance des données stricte doit être établie pour documenter le lignage des données (data lineage) et garantir l'explicabilité du modèle, évitant ainsi toute discrimination algorithmique lors des décisions de crédit.

# Étape 1 : Cadrage du Projet — Définition, Collecte et Analyse des Caractéristiques

---

## 1. Énoncé du problème

L'objectif de ce projet est de prédire la probabilité qu'un emprunteur ne rembourse pas son crédit — la variable cible étant le **statut de défaut (`Loan_Status` : Y pour Oui / N pour Non)** —, afin d'éclairer et d'optimiser la décision d'octroi ou de refus de prêt par l'institution financière.

---

## 2. Plan de collecte de données : Types et Utilité

Pour bâtir un modèle prédictif performant, nous devons segmenter et analyser plusieurs catégories de variables présentes dans notre jeu de données :

* **Données sociodémographiques et situation professionnelle (`Gender`, `Married`, `Dependents`, `Education`, `Self_Employed`) :** Ces informations permettent d'évaluer la stabilité globale de la situation de l'emprunteur, son niveau d'insertion professionnelle et la régularité potentielle de ses rentrées d'argent futures.
* **Données financières et de revenus (`ApplicantIncome`, `CoapplicantIncome`) :** Le niveau de revenus du demandeur principal et du co-demandeur permet de mesurer directement la surface financière brute disponible pour couvrir les charges du foyer.
* **Historique de crédit (`Credit_History`) :** Les antécédents de paiement (respect des échéances passées ou incidents de remboursement) constituent l'indicateur historique le plus puissant pour anticiper le comportement de paiement futur d'un individu.
* **Caractéristiques du prêt demandé (`LoanAmount`, `Loan_Amount_Term`) :** Le montant du prêt et sa durée d'amortissement déterminent directement la lourdeur de la mensualité future ; un déséquilibre face aux revenus augmente mécaniquement le risque de défaut.
* **Garantie et contexte géographique (`Property_Area`) :** La localisation du bien immobilier (Urban, Semiurban, Rural) influence la valeur de la garantie (collatéral) et reflète souvent des disparités économiques locales.

---

## 3. Sources de données et intégration en situation réelle

Dans un cadre de production bancaire, l'alimentation de ce modèle nécessite le croisement de trois sources majeures :
1. **Le système d'information interne (Core Banking) :** Contient l'historique transactionnel et le profil des clients existants. L'intégration se fait via des requêtes SQL automatisées sur le Data Warehouse de l'institution.
2. **Les Bureaux de crédit (Credit Bureaus / Centrales des risques) :** Fournissent l'historique de crédit consolidé au niveau national (`Credit_History`). L'accès s'effectue par des appels d'API sécurisés en temps réel lors de l'étude du dossier.
3. **Le formulaire de demande (Données déclaratives) :** Rempli directement par l'utilisateur (revenus, situation familiale). Ces données sont extraites des bases applicatives (SQL/NoSQL) liées à l'interface client.

---

## 4. Analyse de la pertinence des caractéristiques (Feature Importance a priori)

En analysant le jeu de données fourni (`train_u6lujuX_CVtuZ9i.csv`), les variables se classent selon leur impact prédictif estimé :

| Variable | Catégorie | Niveau de pertinence | Justification comportementale / Métier |
| :--- | :--- | :--- | :--- |
| **`Credit_History`** | Historique | **Critique (N°1)** | Un antécédent négatif (`0`) ou absent est statistiquement le plus fort signal d'exclusion ou de risque de défaut immédiat. |
| **`ApplicantIncome` / `CoapplicantIncome`** | Financier | **Élevé** | Détermine la capacité de remboursement. *Piste d'ingénierie : créer une variable `Total_Income`*. |
| **`LoanAmount` / `Loan_Amount_Term`** | Prêt | **Élevé** | Permet de calculer la charge de la dette (mensualité estimée) par rapport aux revenus globaux. |
| **`Education`** | Socio-éco | **Modéré** | Les profils diplômés (`Graduate`) ont statistiquement accès à des carrières plus stables face aux crises économiques. |
| **`Self_Employed`** | Socio-éco | **Modéré** | Les indépendants présentent des revenus plus volatils, augmentant le risque de liquidité à court terme. |
| **`Dependents`** | Démographie | **Modéré** | Un nombre élevé de personnes à charge réduit mécaniquement le "reste à vivre" après paiement de la mensualité. |
| **`Property_Area`** | Contexte | **Faible à Modéré** | Indique la liquidité et la valeur marchande du bien en cas de saisie ou de recouvrement. |
| **`Married` / `Gender`** | Démographie | **À manipuler avec prudence** | Bien que statistiquement corrélés à une stabilité dans certains anciens modèles, ils posent des questions d'éthique et de biais. |

---

## 5. Risques, contraintes et gouvernance

Le succès de ce modèle repose sur la gestion stricte de plusieurs risques techniques et réglementaires :
* **Confidentialité et Réglementation :** Le traitement de données financières et personnelles (PII) impose une conformité rigoureuse avec les lois de protection des données (RGPD ou équivalents locaux). Cela exige l'anonymisation des identifiants (`Loan_ID`) et le chiffrement des pipelines de données.
* **Qualité des données et déséquilibre :** Le dataset réel comporte des valeurs manquantes (ex: `LoanAmount` ou `Credit_History` non renseignés) et souffre d'un fort déséquilibre des classes (les cas de défaut `N` sont minoritaires), ce qui nécessitera des techniques de nettoyage et d'échantillonnage adaptées (ex: SMOTE ou ajustement des poids des classes).
* **Biais d'échantillonnage :** Nos données historiques ne décrivent que les individus dont le prêt a *déjà* été accepté par le passé. Le modèle aura donc un biais naturel et pourrait avoir du mal à évaluer correctement les profils atypiques ou rejetés historiquement.
* **Glavis et Éthique :** L'utilisation de critères comme le genre (`Gender`) ou la situation matrimoniale (`Married`) peut induire des biais d'équité algorithmique. Une gouvernance stricte doit être mise en place pour garantir l'explicabilité du modèle (via des outils comme SHAP ou LIME) afin de pouvoir justifier un refus de prêt de manière transparente et non discriminatoire.
## 6. Justification approfondie du choix des caractéristiques (Features)

Pour concevoir un modèle de notation (credit scoring) robuste et éviter le phénomène de surapprentissage (*overfitting*), chaque variable sélectionnée doit être justifiée par une logique métier (financière ou comportementale) couplée à une intuition statistique. Voici la rationalisation de notre sélection :

### A. La dimension comportementale : L'historique de crédit
* **`Credit_History` :** En théorie des jeux et en analyse financière, le comportement passé est le meilleur prédicteur du comportement futur. Un emprunteur ayant déjà respecté ses engagements (`1`) démontre une régularité et une volonté de remboursement. Statistiquement, l'absence d'historique ou un antécédent négatif (`0`) multiplie drastiquement le risque de défaut. C'est la variable pivot qui sépare le plus nettement les classes de notre variable cible `Loan_Status`.

### B. La dimension de solvabilité : Les variables financières
* **`ApplicantIncome` & `CoapplicantIncome` :** Un crédit se rembourse grâce aux flux de trésorerie entrants. Analyser le revenu du demandeur principal seul serait une erreur si un co-demandeur (conjoint, associé) apporte une seconde source de revenus. La combinaison de ces deux variables (`Total_Income`) représente l'assiette financière brute sur laquelle l'institution peut s'appuyer en cas de coup dur.
* **`LoanAmount` & `Loan_Amount_Term` :** Le risque de défaut n'est pas absolu, il est relatif à la dette contractée. Le montant total (`LoanAmount`) couplé à la durée (`Loan_Amount_Term`) permet de calculer la mensualité théorique. Une charge de dette trop lourde par rapport aux revenus étouffe financièrement l'emprunteur, élevant mathématiquement la probabilité d'un défaut de paiement par manque de liquidités.

### C. La dimension de stabilité : Les facteurs socio-économiques
* **`Education` & `Self_Employed` :** Ces deux caractéristiques servent d'indicateurs de la résilience du profil face aux chocs économiques. Un diplômé (`Graduate`) possède un capital humain qui réduit la durée de sa recherche d'emploi en cas de licenciement. À l'inverse, un travailleur indépendant (`Self_Employed`), bien que potentiellement très aisé, fait face à une volatilité de revenus intrinsèque à son activité, ce qui augmente le risque systémique à court terme.
* **`Property_Area` :** Cette variable offre un double intérêt. D'une part, elle sert d'indicateur du coût de la vie local (les dynamiques financières en zone urbaine, semi-urbaine ou rurale diffèrent). D'autre part, elle qualifie la valeur et la liquidité du bien immobilier si l'institution doit engager une procédure de saisie ou de recouvrement pour compenser le défaut.

### D. La dimension des charges fixes : La structure familiale
* **`Dependents` :** Le revenu brut ne reflète pas le pouvoir d'achat réel. Un célibataire sans enfant n'a pas les mêmes obligations financières qu'un chef de famille avec plus de trois personnes à charge (`3+`). Plus ce nombre est élevé, plus les dépenses incompressibles (santé, alimentation, scolarité) sont prioritaires sur le remboursement d'un crédit, agissant comme un amplificateur de risque en cas de baisse de revenus.

# Phase 2 : Choix du Modèle et Stratégie d'Évaluation des Performances

---

## 1. Choix du Modèle de Machine Learning

Pour ce projet de prédiction des défauts de paiement (`Loan_Status` = Y/N), nous faisons face à un problème classique de **classification binaire**.

### Le Modèle Principal Privilégié : La Régression Logistique
La **Régression Logistique** est le choix idéal et le modèle de référence à implémenter en priorité pour ce projet, pour plusieurs raisons fondamentales :
* **Interprétabilité et Transparence Métier (L'atout majeur) :** Contrairement aux modèles de type "boîte noire", la Régression Logistique permet d'extraire des coefficients clairs pour chaque variable. Dans le secteur bancaire, il est obligatoire de pouvoir justifier légalement et logiquement un refus de prêt auprès d'un client ou d'un régulateur (ex: *"Le prêt a été refusé principalement à cause d'un historique de crédit égal à 0"*).
* **Adaptation à la Classification :** Elle utilise une fonction sigmoïde pour transformer une combinaison linéaire de caractéristiques en une probabilité mathématique d'appartenir à la classe positive (probabilité de remboursement comprise entre 0 et 1).
* **Simplicité et Performance :** Elle s'entraîne instantanément dans Google Colab, consomme très peu de ressources et s'avère extrêmement robuste lorsque les relations principales du dataset sont linéaires.

### Le Modèle de Référence (Benchmark) : Le Random Forest
Pour valider et challenger la performance de la Régression Logistique, il est stratégique de l'évaluer face à une **Forêt Aléatoire (Random Forest)**. Ce modèle d'ensemble basé sur de multiples arbres de décision permet de :
* Capturer des interactions non linéaires complexes entre les variables (ex: *Si le revenu est faible ET que le demandeur est non-diplômé, le risque augmente de manière exponentielle*).
* Servir de point de comparaison pour s'assurer que notre modèle principal ne passe pas à côté de motifs complexes.

---

## 2. Étapes permettant d'évaluer les performances du modèle

Pour garantir que notre modèle sera capable de se généraliser efficacement sur de futurs clients (données réelles), nous suivrons un protocole rigoureux en 4 étapes :

1. **Séparation des données (Train/Test Split) :** Nous diviserons le dataset initial (`train_u6lujuX_CVtuZ9i.csv`) en deux sous-ensembles distincts : **80% pour le jeu d'entraînement** (apprentissage des motifs) et **20% pour le jeu de test** (données "invisibles" qui servent uniquement à la validation finale).
2. **Validation Croisée (Cross-Validation K-Fold) :** Afin de s'assurer que les performances du modèle ne dépendent pas d'un découpage initial chanceux, nous appliquerons une validation croisée à 5 plis (*5-Fold*) sur le jeu d'entraînement.
3. **Entraînement et Prédiction :** Les algorithmes sont ajustés sur le jeu d'entraînement, puis soumis au jeu de test pour générer des prédictions de probabilité et de classe.
4. **Analyse de la Matrice de Confusion :** Les résultats prédits seront croisés avec les résultats réels du jeu de test pour calculer les indicateurs spécifiques.

---

## 3. Indicateurs de Performance Spécifiques (Métriques)

L'exactitude globale (*Accuracy*) est une métrique trompeuse dans le domaine du crédit car le dataset est souvent déséquilibré (il y a beaucoup plus de prêts accordés que de défauts). Nous utiliserons donc les indicateurs spécifiques suivants, issus de la matrice de confusion :

* **Rappel (Recall / Sensibilité) :** C'est la proportion de cas de défauts réels que le modèle a réussi à identifier correctement. **Pour une institution financière, c'est la métrique la plus critique.** Un faible rappel signifie que le modèle laisse passer des profils à fort risque de non-remboursement, ce qui engendre des pertes financières directes et lourdes pour la banque.
* **Précision (Precision) :** C'est la proportion de prédictions de "Bons payeurs" qui se révèlent exactes en réalité. Une bonne précision garantit que l'institution ne refuse pas à tort des prêts à des clients qui auraient pourtant été solvables (évitant ainsi un manque à gagner commercial).
* **F1-Score :** C'est la moyenne harmonique de la Précision et du Rappel. Il fournit un indicateur unique et équilibré de la performance du modèle, particulièrement adapté aux classes minoritaires (les cas de défaut).
* **Aire Sous la Courbe ROC (AUC-ROC) :** Ce score (compris entre 0 et 1) mesure la capacité globale du modèle à distinguer et séparer les profils sains des profils à risque, quel que soit le seuil de décision statistique choisi. Un score proche de 1 indique un modèle discriminant parfait.

---



# Atelier : Association des Problématiques Métiers aux Modèles de Machine Learning

Ce document résume l'analyse des cas d'usage industriels et justifie le choix des familles d'algorithmes adaptées à chaque situation.

---

## Case Study 1 : Prévision des cours boursiers
* **Objectif :** Prédire le prix futur d'une action (valeur continue).
* **Type de Problème :** Régression (Apprentissage Supervisé).
* **Modèle Sélectionné :** **Random Forest Regressor** (ou Régression Linéaire).
* **Justification :** Le prix d'un actif financier dépend de nombreuses variables macroéconomiques interconnectées. Le Random Forest permet de créer un ensemble d'arbres décisionnels capables de modéliser les fluctuations non linéaires des marchés financiers de manière robuste.

---

## Case Study 2 : Organisation automatique d'une bibliothèque
* **Objectif :** Regrouper des livres en fonction de la similarité de leur contenu ou résumé.
* **Type de Problème :** Clustering / Segmentation (Apprentissage Non Supervisé).
* **Modèle Sélectionné :** **K-Means** (ou algorithmes de Topic Modeling comme LDA).
* **Justification :** Contrairement à la Régression Linéaire (qui calcule une trajectoire numérique), nous n'avons pas ici de variable cible à prédire. L'algorithme doit analyser le texte des livres, calculer une distance mathématique entre les mots, et regrouper les ouvrages similaires au sein de grappes (*clusters*) homogènes.

---

## Case Study 3 : Navigation d'un robot dans un labyrinthe
* **Objectif :** Trouver de manière autonome le chemin le plus court vers la sortie.
* **Type de Problème :** Apprentissage par Renforcement (Reinforcement Learning).
* **Modèle Sélectionné :** **Q-Learning** ou Deep Q-Networks (DQN).
* **Justification :** Un arbre de décision classique est trop statique pour cet environnement changeant. Le robot doit interagir avec son environnement en temps réel : il reçoit une récompense positive lorsqu'il se rapproche de la sortie et une pénalité s'il percute un obstacle. Le modèle apprend ainsi une politique d'action optimale par essais et erreurs.

# Phase 3 : Exploration des Paradigmes de l'Apprentissage Automatique et Stratégies d'Évaluation Advanced

Ce document dresse un cadre méthodologique complet pour l'implémentation, l'évaluation et la gestion des limites des trois grandes familles de l'apprentissage automatique.

---

## 1. Apprentissage Supervisé : Modèle de Classification (Ex: Régression Logistique / Random Forest)

L'apprentissage supervisé s'appuie sur des données historiques étiquetées pour apprendre à associer des caractéristiques (features) à une variable cible connue (ex: `Loan_Status`).

### Stratégie d'Évaluation des Performances
Pour mesurer l'efficacité de notre classifieur de score de crédit, nous appliquons un protocole d'évaluation robuste :

* **Méthodes de Validation :**
    * **Validation Croisée (K-Fold Cross-Validation) :** Le jeu de données d'entraînement est divisé en $K$ sous-ensembles (plis). Le modèle est entraîné $K$ fois sur $K-1$ plis et validé sur le pli restant. Cela permet de s'assurer que la performance est stable et ne dépend pas d'un découpage initial chanceux.
    * **Courbe ROC et score AUC (Area Under the Curve) :** La courbe ROC trace le taux de vrais positifs en fonction du taux de faux positifs pour chaque seuil de probabilité. L'AUC mesure la capacité du modèle à discriminer les classes (un score de 1.0 est parfait, 0.5 équivaut à un choix aléatoire).

* **Choix des Métriques :**
    * **Exactitude (Accuracy) :** $\frac{\text{Vrais Positifs} + \text{Vrais Négatifs}}{\text{Total}}$. Pertinente uniquement si les classes sont parfaitement équilibrées.
    * **Précision (Precision) :** $\frac{\text{Vrais Positifs}}{\text{Vrais Positifs} + \text{Faux Positifs}}$. Mesure la fiabilité d'une prédiction positive (éviter de valider un prêt pour un client insolvable).
    * **Rappel (Recall / Sensibilité) :** $\frac{\text{Vrais Positifs}}{\text{Vrais Positifs} + \text{Faux Négatifs}}$. Mesure la capacité à capter tous les cas réels de défaut. **C'est la métrique financière la plus critique.**
    * **Score F1 :** $2 \times \frac{\text{Précision} \times \text{Rappel}}{\text{Précision} + \text{Rappel}}$. C'est la moyenne harmonique qui synthétise le compromis idéal entre précision et rappel sur des classes déséquilibrées.

### Difficultés et Limites
Le principal défi réside dans le **déséquilibre des classes** (ex: 80% de bons payeurs pour 20% de défauts). Un modèle naïf pourrait obtenir 80% d'exactitude en acceptant tout le monde, tout en étant catastrophique pour la banque. De plus, la dérive des données (*Data Drift*) peut rendre le modèle obsolète si les conditions économiques changent après son déploiement.

---

## 2. Apprentissage Non Supervisé : Modèle de Clustering (Ex: K-Means)

L'apprentissage non supervisé cherche à découvrir des structures cachées ou des regroupements naturels (*clusters*) au sein de données non étiquetées (ex: regrouper des livres par thématiques ou segmenter des profils clients).



### Stratégie d'Évaluation de l'Efficacité
N'ayant pas de "vérité terrain" (étiquettes) pour valider les groupes, l'évaluation repose sur des critères géométriques de proximité :

* **La Méthode du Coude (Elbow Method) :** On calcule l'inertie intra-classe (la somme des carrés des distances entre chaque point et le centre de son cluster) pour différents nombres de clusters $K$. En traçant la courbe, on observe une cassure (un "coude"). Le point d'inflexion indique le nombre optimal de clusters, là où l'ajout d'un groupe supplémentaire n'apporte plus de gain de compacité significatif.
* **Le Score de Silhouette :** Mesure à quel point un objet est similaire à son propre cluster par rapport aux autres clusters. Le score varie entre -1 et 1. Un score proche de 1 signifie que l'objet est parfaitement positionné dans son groupe et très éloigné des groupes voisins.
* **Métriques de Validation par Cluster :** Analyse de la densité intrinsèque des groupes formés et de la séparation maximale entre les frontières de chaque cluster.

### Difficultés et Limites
L'évaluation est intrinsèquement **subjective**. Un découpage mathématiquement parfait (basé sur les distances) peut n'avoir aucune valeur ou signification métier réelle. De plus, les algorithmes de clustering comme K-Means sont extrêmement sensibles aux valeurs aberrantes (*outliers*) et à la mise à l'échelle des données (*feature scaling*), nécessitant une normalisation obligatoire (ex: `StandardScaler`).

---

## 3. Apprentissage par Renforcement (Ex: Q-Learning / DQN)

Dans ce paradigme, un agent autonome interagit de manière dynamique avec un environnement. Il apprend à prendre des décisions séquentielles en maximisant un signal de récompense au fil du temps (ex: un robot apprenant à sortir d'un labyrinthe).

### Stratégie de Mesure du Succès
Le succès de l'apprentissage ne se mesure pas sur une base fixe, mais sur la trajectoire temporelle de l'agent :

* **Récompense Cumulative (Cumulative Reward) :** On suit l'évolution de la somme des récompenses obtenues par l'agent au cours de chaque épisode. Une courbe de récompense cumulative qui croît de manière constante prouve que l'agent adopte un comportement de plus en plus optimal.
* **La Convergence :** C'est le moment où la politique de l'agent se stabilise. On considère que le modèle a convergé lorsque la variation des fonctions de valeur (comme la Q-table) ou le taux de réussite de la tâche (ex: atteindre la sortie) se stabilisent à un niveau maximal sur plusieurs épisodes successifs.
* **Équilibre Exploration / Exploitation :** Mesuré à travers le suivi de paramètres dynamiques comme le taux $\epsilon$-greedy ($\epsilon$). En début d'apprentissage, $\epsilon$ est élevé (le robot explore au hasard pour découvrir des chemins). Au fil du temps, $\epsilon$ diminue (décroissance de l'exploration) pour laisser l'agent *exploiter* les chemins optimaux qu'il a mémorisés.

### Difficultés et Limites
La principale limite est la **conception de la fonction de récompense** (*Reward Shaping*). Si elle est mal calibrée, l'agent peut trouver des failles mathématiques pour maximiser ses points sans remplir l'objectif réel. De plus, l'apprentissage par renforcement souffre d'une instabilité chronique, d'un coût de calcul immense, et d'un problème d'attribution temporelle du crédit (savoir quelle action spécifique menée au début a provoqué la victoire ou la défaite à la fin).